# 09 — Modelo LightGBM: predicción individual por estación

**Proyecto:** RepostaPro — Optimización del repostaje en flotas comerciales  
**Autor:** Víctor González Martín  
**Notebook:** 09 — LightGBM como modelo de producción

## Objetivo del notebook

Entrenar y evaluar LightGBM como modelo de gradient boosting tabular sobre los datasets analíticos generados. A diferencia de Prophet (que modela una serie temporal nacional), LightGBM predice **el precio individual de cada estación** aprovechando las 18 features generadas.

## Características de LightGBM

- **Algoritmo**: gradient boosting sobre árboles de decisión optimizado.
- **Velocidad**: implementación con histogramas y leaf-wise growth.
- **Categóricas nativas**: maneja Rotulo_normalizado, Provincia, IDCCAA sin codificación one-hot.
- **Valores NaN nativos**: no requiere imputación de lags y medias móviles iniciales.
- **Early stopping**: previene overfitting mediante monitorización del conjunto de validación.

## Configuración

- **Features**: 18 variables (calendario, régimen, lags, medias móviles, contexto nacional, categóricas geográficas).
- **Hiperparámetros base**: configuración razonable para regresión tabular sin tuning específico todavía.
- **Early stopping**: 30 rondas sin mejora en validation → para entrenamiento.

## Modelos a entrenar

4 modelos en total (2 carburantes × 2 régimenes):

| Modelo | Carburante | Régimen | Filas train |
|---|---|---|---|
| 1 | Gasóleo A | Pre-shock | ~6,99 M |
| 2 | Gasóleo A | Post-shock | ~687 K |
| 3 | Gasolina 95 E5 | Pre-shock | ~6,76 M |
| 4 | Gasolina 95 E5 | Post-shock | ~665 K |

In [1]:
# Configuración del entorno
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.family"] = "Calibri"
plt.rcParams["font.size"] = 11
sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Añadir raíz del proyecto al sys.path
RAIZ_PROYECTO = Path("..").resolve()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

# Importar funciones de los módulos
from src.modelos import (cargar_todas_particiones,preparar_datos_lightgbm,entrenar_lightgbm,predecir_lightgbm,evaluar_lightgbm_en_conjunto,
    imprimir_evaluacion,guardar_modelo_lightgbm,importancia_features_lightgbm,evaluar,FEATURES_LIGHTGBM,)

# Rutas
CARPETA_PARTICIONES = Path("../data/processed/particiones")
CARPETA_MODELOS = Path("../outputs/models/lightgbm")
CARPETA_RESULTADOS = Path("../outputs/resultados_modelos")
CARPETA_FIGURAS = Path("../outputs/figures")
CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {RAIZ_PROYECTO}")
print(f"Carpeta de modelos:    {CARPETA_MODELOS.resolve()}")
print(f"Carpeta de resultados: {CARPETA_RESULTADOS.resolve()}")
print(f"\nFeatures de entrada: {len(FEATURES_LIGHTGBM)}")
for i, f in enumerate(FEATURES_LIGHTGBM, 1):
    print(f"  {i:2}. {f}")
print(f"\nMódulos importados correctamente")

Raíz del proyecto: C:\TFM
Carpeta de modelos:    C:\TFM\outputs\models\lightgbm
Carpeta de resultados: C:\TFM\outputs\resultados_modelos

Features de entrada: 18
   1. dia_semana
   2. mes
   3. dia_del_mes
   4. semana_del_año
   5. es_fin_semana
   6. es_festivo_nacional
   7. regimen
   8. dias_desde_shock
   9. precio_lag_1
  10. precio_lag_7
  11. precio_lag_30
  12. precio_mm_7
  13. precio_mm_30
  14. precio_medio_nacional_dia
  15. diferencial_vs_nacional
  16. Rotulo_normalizado
  17. Provincia
  18. IDCCAA

✓ Módulos importados correctamente


## Función pipeline LightGBM

Encapsulamos en una función reutilizable todo el flujo: carga de particiones, preparación de datos, entrenamiento con early stopping, evaluación en train/val/test y guardado del modelo.

In [2]:
def pipeline_lightgbm(carburante, regimen):
    """Pipeline completo LightGBM: entrena, evalúa y guarda.

    Args:
        carburante: 'gasoleo_a' o 'gasolina_95'.
        regimen: 'pre_shock' o 'post_shock'.

    Returns:
        dict con modelo, métricas y particiones.
    """
    nombre_completo = f"{carburante.upper()} · {regimen.replace('_', ' ')}"
    print(f"\n>>> Entrenando LightGBM: {nombre_completo}")
    print("=" * 70)
    
    # 1. Cargar particiones
    inicio_carga = time.time()
    particiones = cargar_todas_particiones(CARPETA_PARTICIONES, carburante, regimen)
    print(f"Particiones cargadas en {time.time() - inicio_carga:.1f}s")
    print(f"  Train: {len(particiones['train']):,} filas")
    print(f"  Val:   {len(particiones['val']):,} filas")
    print(f"  Test:  {len(particiones['test']):,} filas")
    
    # 2. Preparar datos train y val (para early stopping)
    X_train, y_train = preparar_datos_lightgbm(particiones["train"])
    X_val, y_val = preparar_datos_lightgbm(particiones["val"])
    
    # 3. Entrenar con early stopping
    print(f"\nEntrenando modelo (early stopping en validation)...")
    inicio_train = time.time()
    
    modelo = entrenar_lightgbm(X_train, y_train, X_val, y_val)
    
    duracion = time.time() - inicio_train
    print(f"✓ Modelo entrenado en {duracion:.1f}s ({duracion/60:.1f} min)")
    print(f"  Iteraciones realizadas: {modelo.best_iteration_}")
    
    # 4. Evaluar en los 3 conjuntos
    resultados = []
    for conjunto in ["train", "val", "test"]:
        resultado = evaluar_lightgbm_en_conjunto(modelo, particiones[conjunto],nombre=f"LightGBM · {nombre_completo} · {conjunto}")
        resultados.append(resultado)
    
    imprimir_evaluacion(resultados)
    
    # 5. Guardar modelo
    ruta_modelo = CARPETA_MODELOS / f"lightgbm_{carburante}_{regimen}.txt"
    guardar_modelo_lightgbm(modelo, ruta_modelo)
    tamano_mb = ruta_modelo.stat().st_size / 1024 / 1024
    print(f"\n✓ Modelo guardado: {ruta_modelo.name} ({tamano_mb:.1f} MB)")
    
    return {
        "carburante": carburante,
        "regimen": regimen,
        "modelo": modelo,
        "resultados": resultados,
        "particiones": particiones}

print("Función pipeline_lightgbm definida")

✓ Función pipeline_lightgbm definida


## Modelo 1: Gasóleo A · Pre-shock

Primer modelo. La combinación más exigente: 6,99 millones de filas de entrenamiento. Tiempo esperado: 5-12 minutos.

In [3]:
resultado_gasoleo_pre = pipeline_lightgbm("gasoleo_a", "pre_shock")


>>> Entrenando LightGBM: GASOLEO_A · pre shock
Particiones cargadas en 2.2s
  Train: 6,996,238 filas
  Val:   1,017,945 filas
  Test:  651,949 filas

Entrenando modelo (early stopping en validation)...
✓ Modelo entrenado en 165.8s (2.8 min)
  Iteraciones realizadas: 499
Conjunto                                   N obs    MAE (cts)   RMSE (cts)     MAPE (%)
------------------------------------------------------------------------------------------
LightGBM · GASOLEO_A · pre shock · train    6,996,238       0.150        0.279        0.106
LightGBM · GASOLEO_A · pre shock · val    1,017,945       0.249        0.395        0.182
LightGBM · GASOLEO_A · pre shock · test      651,949       0.456        0.570        0.330

✓ Modelo guardado: lightgbm_gasoleo_a_pre_shock.txt (3.3 MB)


## Modelo 2: Gasóleo A · Post-shock

Régimen tras el shock geopolítico. Dataset más pequeño (687 K filas) pero régimen más volátil.

In [4]:
resultado_gasoleo_post = pipeline_lightgbm("gasoleo_a", "post_shock")


>>> Entrenando LightGBM: GASOLEO_A · post shock
Particiones cargadas en 0.3s
  Train: 687,425 filas
  Val:   269,279 filas
  Test:  236,037 filas

Entrenando modelo (early stopping en validation)...
✓ Modelo entrenado en 6.6s (0.1 min)
  Iteraciones realizadas: 90
Conjunto                                   N obs    MAE (cts)   RMSE (cts)     MAPE (%)
------------------------------------------------------------------------------------------
LightGBM · GASOLEO_A · post shock · train      687,425       0.553        0.907        0.322
LightGBM · GASOLEO_A · post shock · val      269,279       2.452        2.731        1.434
LightGBM · GASOLEO_A · post shock · test      236,037       6.639        7.089        4.093

✓ Modelo guardado: lightgbm_gasoleo_a_post_shock.txt (0.6 MB)


## Modelos 3 y 4: Gasolina 95 E5 · Ambos régimenes

In [5]:
resultado_gasolina_pre = pipeline_lightgbm("gasolina_95", "pre_shock")
resultado_gasolina_post = pipeline_lightgbm("gasolina_95", "post_shock")


>>> Entrenando LightGBM: GASOLINA_95 · pre shock
Particiones cargadas en 2.2s
  Train: 6,759,112 filas
  Val:   985,129 filas
  Test:  631,170 filas

Entrenando modelo (early stopping en validation)...
✓ Modelo entrenado en 67.7s (1.1 min)
  Iteraciones realizadas: 168
Conjunto                                   N obs    MAE (cts)   RMSE (cts)     MAPE (%)
------------------------------------------------------------------------------------------
LightGBM · GASOLINA_95 · pre shock · train    6,759,112       0.269        0.508        0.177
LightGBM · GASOLINA_95 · pre shock · val      985,129       0.534        0.796        0.371
LightGBM · GASOLINA_95 · pre shock · test      631,170       0.872        1.123        0.607

✓ Modelo guardado: lightgbm_gasolina_95_pre_shock.txt (1.1 MB)

>>> Entrenando LightGBM: GASOLINA_95 · post shock
Particiones cargadas en 0.3s
  Train: 664,995 filas
  Val:   260,617 filas
  Test:  228,394 filas

Entrenando modelo (early stopping en validation)...
✓ Mod